# LEI lookup: one Legal Entity Identifier, looked up in the Global LEI Index

Runs [LEI Lookup: Legal Entity Identifier Search by Name or Status](https://apify.com/yappman/lei-lookup) in LEI mode with its prefilled
input: a single LEI. Each row is one entity from the daily Golden Copy of the Global LEI
Index, in one fixed schema designed not to include individuals' details: the legal name, registration and entity
status, jurisdiction, legal form, a country-region-city address and the next renewal date.
The notebook shows the row as a table and reads the run summary, which lists anything
that was refused or not found.

**Cost at Apify Free-tier prices:** at most **$0.001** per run (one entity at $0.001; there
is no start fee). An LEI that is invalid or not found costs nothing. In LEI mode the list
of LEIs is the cap, because `maxItems` applies only to name and status searches;
`max_total_charge_usd` caps the bill.

More data: [data.yappman.com](https://data.yappman.com).

In [ ]:
import os
from datetime import timedelta
from decimal import Decimal
import time

import pandas as pd
from apify_client import ApifyClient

TOKEN = os.environ.get("APIFY_TOKEN")
if not TOKEN:
    raise RuntimeError("Set the APIFY_TOKEN environment variable first. See the README.")
client = ApifyClient(TOKEN)
pd.set_option("display.html.use_mathjax", False)  # show third-party text as text


def run_actor(actor_id: str, run_input: dict, max_charge_usd: float):
    """Start an Actor, wait for it, and return the finished run.

    Spend is capped by the size of the input and by max_total_charge_usd, which
    stops the run if its charges would pass the cap. The run times out after 10 minutes.
    If a run is not SUCCEEDED, check it in the Console before running the cell again.
    """
    run = client.actor(actor_id).call(
        run_input=run_input,
        max_total_charge_usd=Decimal(str(max_charge_usd)),
        run_timeout=timedelta(minutes=10),
    )
    if run is None or run.status != "SUCCEEDED":
        status = None if run is None else run.status
        where = "" if run is None else f" Run: https://console.apify.com/actors/runs/{run.id}"
        raise RuntimeError(f"{actor_id} did not succeed: {status}.{where}")
    print(f"{actor_id}: run {run.id}")
    # Charges settle a few seconds after the run ends, so read the record again before printing them.
    time.sleep(5)
    settled = client.run(run.id).get()
    print("Charged events:", settled.charged_event_counts)
    return run

## 1. Run the lookup

Each LEI is checked as ISO 17442 before any request. A malformed one is refused on its own
and costs nothing.

In [ ]:
ACTOR = "yappman/lei-lookup"
RUN_INPUT = {
    "mode": "lei",
    "leis": ["2138002P5RNKC5W2JZ46"],
}

# Free tier: 1 entity x $0.001 = $0.001
run = run_actor(ACTOR, RUN_INPUT, max_charge_usd=0.005)
rows = list(client.dataset(run.default_dataset_id).iterate_items())
print(len(rows), "entities")

## 2. The entity

Rows come back in input order. Where the entity is a person, such as a sole proprietor,
the name, the register number and the addresses are withheld and listed in `withheld`;
the LEI, the statuses and the dates are still returned.

In [ ]:
def place(address):
    if not isinstance(address, dict):
        return None
    return ", ".join(str(address[k]) for k in ("city", "region", "country") if address.get(k))


df = pd.DataFrame(rows)
if df.empty:
    print("No entities returned.")
else:
    df = df.reindex(columns=["lei", "legalName", "registrationStatus", "entityStatus", "jurisdiction",
                             "legalForm", "legalAddress", "nextRenewalDate", "withheld", "goldenCopyDate"])
    df["legalAddress"] = df["legalAddress"].apply(place)
    display(df)

## 3. The run summary

Anything refused or not found is listed here instead of in the table, and none of it is
charged.

In [ ]:
record = client.key_value_store(run.default_key_value_store_id).get_record("RUN_SUMMARY")
summary = (record or {}).get("value") or {}
print({k: summary.get(k) for k in
       ("status", "mode", "pushed", "found", "requested", "notFound", "refused",
        "unavailable", "withheldPersons", "goldenCopyDate")})

## Notes

* The data is the Global LEI Index published by GLEIF through its API, under CC0
  per GLEIF's LEI Data Terms of Use. No attribution is required. This is not affiliated
  with, or endorsed by, GLEIF or any LEI issuer.
* Addresses are country, region and city only. No street, number or postcode is read.
* Rows belong to the Actor's data terms, not to this repository's MIT licence.